# Quarantine_And_Alert_Execution

Runs when the quality gate fails. Writes staged rejects to the permanent quarantine table and reports the count for the pipeline's alert step.

In [1]:
from pyspark.sql import functions as F
from notebookutils import mssparkutils

quarantine_table_name = "quarantine_orders"

print("=" * 60)
print(" QUALITY GATE FAILED: QUARANTINING ENTIRE FILE ")
print("=" * 60)

# orders_pending_routing is written by Orders_Bronze_to_Silver in the same
# pipeline run. A failed file-level quality gate means the whole batch is
# not trusted — so every row (valid and invalid alike) goes to quarantine,
# not just the invalid ones.
if not spark.catalog.tableExists("orders_pending_routing"):
    raise ValueError(
        "Table 'orders_pending_routing' was not found. "
        "Run Orders_Bronze_to_Silver first."
    )

df_full = spark.read.table("orders_pending_routing")

StatementMeta(, b70e41c7-fa4c-42df-b9b1-ae14616a80ea, 3, Finished, Available, Finished, False)

 QUALITY GATE FAILED: QUARANTINING ENTIRE FILE 


In [2]:
# Write to quarantine — a permanent audit trail across all runs.
df_full = (
    df_full.withColumn("QuarantinedAt", F.current_timestamp())
    .withColumn("OverallReason", F.lit("file_quality_gate_failed"))
)
df_full.write.format("delta").mode("append").saveAsTable(quarantine_table_name)

# Report the count and signal completion to the pipeline.
quarantine_count = df_full.count()
print(f"-> ALERT: {quarantine_count} row(s) — full file — quarantined due to quality gate failure.")

mssparkutils.notebook.exit('{"status": "quarantine_complete"}')

StatementMeta(, b70e41c7-fa4c-42df-b9b1-ae14616a80ea, 4, Finished, Available, Finished, False)

-> ALERT: 30 row(s) — full file — quarantined due to quality gate failure.
ExitValue: {"status": "quarantine_complete"}